# Chatbot using Transformers

This notebook builds a simple conversational chatbot using a pre-trained Hugging Face transformer model (`microsoft/DialoGPT-small`).

- Loading a pre-trained transformer model
- Taking user input
- Generating model responses
- Maintaining multi-turn conversation context
- Exiting chat with `exit` or `quit`

In [1]:
# Install required dependencies (run once).
%pip install torch transformers --quiet

Note: you may need to restart the kernel to use updated packages.


In [2]:
import re
import torch
import requests
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

print(f'Torch version: {torch.__version__}')

Torch version: 2.11.0+cpu


In [3]:
# Load an instruction-tuned transformer model from Hugging Face.
MODEL_NAME = 'google/flan-t5-large'

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

# Use CPU for compatibility in most student environments.
device = torch.device('cpu')
model = model.to(device)

print(f'Model loaded: {MODEL_NAME}')

config.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

c:\Users\DELL\anaconda3\envs\mlenv\lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\DELL\.cache\huggingface\hub\models--google--flan-t5-large. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/3.13G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Model loaded: google/flan-t5-large


In [4]:
def get_wikipedia_context(query):
    """Fetch short factual context from Wikipedia REST APIs."""
    try:
        search_resp = requests.get(
            'https://en.wikipedia.org/w/api.php',
            params={
                'action': 'opensearch',
                'search': query,
                'limit': 1,
                'namespace': 0,
                'format': 'json'
            },
            timeout=8
        )
        search_resp.raise_for_status()
        search_data = search_resp.json()

        titles = search_data[1] if len(search_data) > 1 else []
        if not titles:
            return ''

        title = titles[0]
        summary_resp = requests.get(
            f'https://en.wikipedia.org/api/rest_v1/page/summary/{title}',
            timeout=8
        )
        summary_resp.raise_for_status()
        summary_data = summary_resp.json()

        return summary_data.get('extract', '')
    except Exception:
        return ''


def get_duckduckgo_context(query):
    """Fallback factual context from DuckDuckGo Instant Answer API."""
    try:
        resp = requests.get(
            'https://api.duckduckgo.com/',
            params={'q': query, 'format': 'json', 'no_html': 1, 'skip_disambig': 1},
            timeout=8
        )
        resp.raise_for_status()
        data = resp.json()

        abstract = data.get('AbstractText', '').strip()
        if abstract:
            return abstract

        related = data.get('RelatedTopics', [])
        for item in related:
            if isinstance(item, dict) and item.get('Text'):
                return item['Text']

        return ''
    except Exception:
        return ''


def get_factual_context(query):
    """Try multiple sources to obtain factual grounding context."""
    context = get_wikipedia_context(query)
    if context:
        return context

    return get_duckduckgo_context(query)


def extract_creator_answer(question_text, context):
    """Try to extract a creator sentence from context for 'who created' type questions."""
    q = question_text.lower()
    if 'who created' not in q and 'who developed' not in q and 'who invented' not in q:
        return ''

    sentences = re.split(r'(?<=[.!?])\s+', context)
    for sentence in sentences:
        lower_sentence = sentence.lower()
        if any(key in lower_sentence for key in ['created by', 'developed by', 'designed by', 'invented by']):
            return sentence.strip()

    return ''


def generate_bot_response(user_text, conversation_history, max_new_tokens=80):
    """Generate one chatbot response with grounded factual answering when possible."""
    cleaned = user_text.strip()
    lower_text = cleaned.lower()
    starts_as_question = lower_text.startswith(('who', 'what', 'when', 'where', 'why', 'how'))
    is_question = cleaned.endswith('?') or starts_as_question

    # Disambiguation rule: in this assignment context, treat Python as the programming language.
    if 'python' in lower_text and 'language' not in lower_text and 'snake' not in lower_text:
        cleaned = cleaned.replace('Python', 'Python programming language').replace('python', 'Python programming language')

    # Keep only the latest few turns for context.
    recent_history = conversation_history[-6:]
    history_block = "\n".join(recent_history)

    if is_question:
        context = get_factual_context(cleaned)

        # Prefer direct extraction for creator-style questions to reduce hallucination.
        extracted_answer = extract_creator_answer(cleaned, context) if context else ''
        if extracted_answer:
            return extracted_answer

        if context:
            prompt = (
                "Use the context to answer accurately in 1-2 sentences.\n"
                f"Context: {context}\n"
                f"Question: {cleaned}\n"
                "Answer:"
            )
        else:
            prompt = (
                "You are a factual technology assistant. "
                "Answer accurately in 1-2 sentences.\n"
                f"Question: {cleaned}\n"
                "Answer:"
            )
    else:
        prompt = (
            "You are a friendly AI assistant for technology learners. "
            "Reply naturally in 1-3 sentences.\n\n"
            f"Conversation so far:\n{history_block}\n"
            f"User: {cleaned}\n"
            "Assistant:"
        )

    inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=512).to(device)

    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        num_beams=5,
        repetition_penalty=1.15,
        no_repeat_ngram_size=3,
        length_penalty=0.9
    )

    bot_text = tokenizer.decode(output_ids[0], skip_special_tokens=True).strip()
    return bot_text

In [5]:
def run_chatbot():
    """Run an interactive chatbot loop in the notebook console."""
    print('Chatbot: Hello! I am your AI assistant. How can I help you today?')

    conversation_history = []

    while True:
        user_message = input('Type your message and press Enter: ').strip()

        # Echo user text so it appears in notebook output transcript.
        print(f'User: {user_message}')

        if user_message.lower() in {'exit', 'quit'}:
            print('Chatbot: Conversation ended. Have a great day!')
            break

        if not user_message:
            print("Chatbot: Please type a message or use 'exit' to quit.")
            continue

        bot_reply = generate_bot_response(user_message, conversation_history)
        print(f'Chatbot: {bot_reply}')

        # Save turns to preserve context for the next response.
        conversation_history.append(f'User: {user_message}')
        conversation_history.append(f'Assistant: {bot_reply}')

In [6]:
# Run the chatbot. Type 'exit' or 'quit' to stop.
run_chatbot()

Chatbot: Hello! I am your AI assistant. How can I help you today?
User: what is the chatbot?
Chatbot: a software application or web interface designed to converse through text or speech
User: what is the use of chatbot?
Chatbot: chatbot is a computer program that automates the interaction between humans and computers.
User: what are the resources are you using to interact with humans?
Chatbot: Human-Computer Interaction
User: exit
Chatbot: Conversation ended. Have a great day!
